In [4]:
%%writefile milestone1.cpp
#include <iostream>
#include <vector>
#include <random>
#include <chrono>
#include <cmath>

// ==========================================
// SHARED DATA STRUCTURES
// ==========================================
struct Point {
    double x, y;
};

struct BoundingBox {
    double minX, minY, maxX, maxY;
    bool contains(const Point& p) const {
        return (p.x >= minX && p.x <= maxX && p.y >= minY && p.y <= maxY);
    }
    bool intersects(const BoundingBox& other) const {
        return !(other.minX > maxX || other.maxX < minX || other.minY > maxY || other.maxY < minY);
    }
};

struct Polygon {
    int id;
    std::vector<Point> vertices;
    BoundingBox bbox;
};

// ==========================================
// TASK: Saad Thaplawala (27172)
// Bounding-box computation and integration
// ==========================================
void computeBoundingBox(Polygon& poly) {
    if (poly.vertices.empty()) return;
    poly.bbox.minX = poly.vertices[0].x;
    poly.bbox.maxX = poly.vertices[0].x;
    poly.bbox.minY = poly.vertices[0].y;
    poly.bbox.maxY = poly.vertices[0].y;

    for (const auto& v : poly.vertices) {
        if (v.x < poly.bbox.minX) poly.bbox.minX = v.x;
        if (v.x > poly.bbox.maxX) poly.bbox.maxX = v.x;
        if (v.y < poly.bbox.minY) poly.bbox.minY = v.y;
        if (v.y > poly.bbox.maxY) poly.bbox.maxY = v.y;
    }
}

// ==========================================
// TASK: Shaikh Muhammad Umair (26409)
// Ray-Casting algorithm for Point-in-Polygon
// ==========================================
bool pointInPolygon(const Point& p, const Polygon& poly) {
    // First, check the bounding box (Saad's pipeline integration)
    if (!poly.bbox.contains(p)) {
        return false;
    }

    bool inside = false;
    int n = poly.vertices.size();
    for (int i = 0, j = n - 1; i < n; j = i++) {
        double xi = poly.vertices[i].x, yi = poly.vertices[i].y;
        double xj = poly.vertices[j].x, yj = poly.vertices[j].y;

        // Ray casting to the right
        bool intersect = ((yi > p.y) != (yj > p.y)) &&
                         (p.x < (xj - xi) * (p.y - yi) / (yj - yi) + xi);
        if (intersect) inside = !inside;
    }
    return inside;
}

// ==========================================
// TASK: Muhammad Ibrahim Farid (27098)
// Spatial Index (Simplified Quadtree)
// ==========================================
class Quadtree {
private:
    static const int MAX_CAPACITY = 4;
    BoundingBox boundary;
    std::vector<Polygon*> polygons;
    bool divided = false;
    Quadtree* nw; Quadtree* ne; Quadtree* sw; Quadtree* se;

    void subdivide() {
        double x = boundary.minX; double y = boundary.minY;
        double w = (boundary.maxX - boundary.minX) / 2.0;
        double h = (boundary.maxY - boundary.minY) / 2.0;

        nw = new Quadtree({x, y + h, x + w, y + h * 2});
        ne = new Quadtree({x + w, y + h, x + w * 2, y + h * 2});
        sw = new Quadtree({x, y, x + w, y + h});
        se = new Quadtree({x + w, y, x + w * 2, y + h});
        divided = true;
    }

public:
    Quadtree(BoundingBox b) : boundary(b), nw(nullptr), ne(nullptr), sw(nullptr), se(nullptr) {}
    ~Quadtree() { delete nw; delete ne; delete sw; delete se; }

    bool insert(Polygon* poly) {
        if (!boundary.intersects(poly->bbox)) return false;

        if (polygons.size() < MAX_CAPACITY) {
            polygons.push_back(poly);
            return true;
        }
        if (!divided) subdivide();

        if (nw->insert(poly)) return true;
        if (ne->insert(poly)) return true;
        if (sw->insert(poly)) return true;
        if (se->insert(poly)) return true;
        return false;
    }

    void query(const Point& p, std::vector<Polygon*>& found) {
        if (!boundary.contains(p)) return;

        for (auto poly : polygons) {
            if (poly->bbox.contains(p)) {
                found.push_back(poly);
            }
        }
        if (divided) {
            nw->query(p, found);
            ne->query(p, found);
            sw->query(p, found);
            se->query(p, found);
        }
    }
};

// ==========================================
// TASK: Fawad Siddiqui (29275)
// Synthetic Data Gen & Validation Framework
// ==========================================
std::vector<Point> generateGPSData(int numPoints, bool isClustered) {
    std::vector<Point> points;
    points.reserve(numPoints);

    std::random_device rd;
    std::mt19937 gen(rd()); // FIXED: Changed to valid standard random engine

    if (isClustered) {
        // Simulating spatial skew (urban hotspots)
        std::normal_distribution<double> dX(50.0, 10.0); // Cluster center X
        std::normal_distribution<double> dY(50.0, 10.0); // Cluster center Y
        for (int i = 0; i < numPoints; ++i) {
            points.push_back(Point{dX(gen), dY(gen)}); // Added explicit Point{} cast
        }
    } else {
        // Uniform distribution
        std::uniform_real_distribution<double> dist(0.0, 100.0);
        for (int i = 0; i < numPoints; ++i) {
            points.push_back(Point{dist(gen), dist(gen)}); // Added explicit Point{} cast
        }
    }
    return points;
}

int bruteForceValidation(const std::vector<Point>& points, const std::vector<Polygon>& polygons) {
    int pointsInside = 0;
    for (const auto& p : points) {
        for (const auto& poly : polygons) {
            if (pointInPolygon(p, poly)) {
                pointsInside++;
                break; // Found its polygon, move to next point
            }
        }
    }
    return pointsInside;
}

// ==========================================
// MAIN EXECUTION
// ==========================================
int main() {
    std::cout << "--- PDC Project: Milestone 1 ---" << std::endl;

    // 1. Generate Test Polygons
    std::vector<Polygon> polygons(2);
    // Polygon 1: A square from (10,10) to (30,30)
    polygons[0] = {1, {Point{10,10}, Point{30,10}, Point{30,30}, Point{10,30}}, {}};
    // Polygon 2: A square from (60,60) to (90,90)
    polygons[1] = {2, {Point{60,60}, Point{90,60}, Point{90,90}, Point{60,90}}, {}};

    for (auto& poly : polygons) computeBoundingBox(poly);

    // 2. Generate Points (Fawad's Task)
    int numPoints = 100000; // Testing with 100k points for baseline
    std::cout << "Generating " << numPoints << " clustered GPS points..." << std::endl;
    std::vector<Point> points = generateGPSData(numPoints, true);

    // 3. Brute Force Validation (Fawad's Task)
    std::cout << "Running Brute Force Validation..." << std::endl;
    auto startBF = std::chrono::high_resolution_clock::now();
    int bfHits = bruteForceValidation(points, polygons);
    auto endBF = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> bfDuration = endBF - startBF;
    std::cout << "Brute Force Hits: " << bfHits << " | Time: " << bfDuration.count() << " seconds\n" << std::endl;

    // 4. Spatial Indexing Pipeline (Ibrahim & Saad's Tasks)
    std::cout << "Building Quadtree and running optimized query..." << std::endl;
    Quadtree qt({0.0, 0.0, 100.0, 100.0});
    for (auto& poly : polygons) {
        qt.insert(&poly);
    }

    auto startQT = std::chrono::high_resolution_clock::now();
    int qtHits = 0;
    for (const auto& p : points) {
        std::vector<Polygon*> candidates;
        qt.query(p, candidates);
        for (auto polyPtr : candidates) {
            if (pointInPolygon(p, *polyPtr)) {
                qtHits++;
                break;
            }
        }
    }
    auto endQT = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> qtDuration = endQT - startQT;

    std::cout << "Quadtree Hits: " << qtHits << " | Time: " << qtDuration.count() << " seconds" << std::endl;

    // 5. Verification
    if (bfHits == qtHits) {
        std::cout << "\nSUCCESS: Brute-force validation matches the Quadtree pipeline!" << std::endl;
    } else {
        std::cout << "\nERROR: Mismatch detected between pipelines." << std::endl;
    }

    return 0;
}

Writing milestone1.cpp


In [5]:
!g++ -O3 milestone1.cpp -o milestone1

In [6]:
!./milestone1

--- PDC Project: Milestone 1 ---
Generating 100000 clustered GPS points...
Running Brute Force Validation...
Brute Force Hits: 2541 | Time: 0.00042296 seconds

Building Quadtree and running optimized query...
Quadtree Hits: 2541 | Time: 0.00078396 seconds

SUCCESS: Brute-force validation matches the Quadtree pipeline!
